In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
FINAL_DIR = DATA_DIR / "oral_seg_final"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

for carpeta in [TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

CSV_DATASET = TABLES_DIR / "oral_seg_dataset_final.csv"

print("TensorFlow:", tf.__version__)
print("CSV existe:", CSV_DATASET.exists())
print("Dataset final:", FINAL_DIR)

print("\nGPUs detectadas por TensorFlow:")
print(tf.config.list_physical_devices("GPU"))

In [ ]:
df = pd.read_csv(CSV_DATASET)

print("Total muestras:", len(df))
display(df.head())

print("\nReparto por split:")
print(df["split"].value_counts())

print("\nReparto por origen:")
print(df["origen"].value_counts())

print("\nReparto por clase:")
print(df["clase_lengua"].value_counts())

In [ ]:
df_train = df[df["split"] == "train"].copy()
df_val = df[df["split"] == "val"].copy()
df_test = df[df["split"] == "test"].copy()

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 20
RANDOM_STATE = 7

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)

In [ ]:
def cargar_imagen_y_mascara(img_path, mask_path):
    """
    Lee una imagen y su máscara.

    Imagen:
    - RGB
    - float32
    - rango 0-1

    Máscara:
    - 1 canal
    - float32
    - valores 0 o 1
    """

    img_bytes = tf.io.read_file(img_path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.io.decode_image(mask_bytes, channels=1, expand_animations=False)
    mask.set_shape([None, None, 1])
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask, tf.float32) / 255.0

    mask = tf.where(mask > 0.5, 1.0, 0.0)

    return img, mask

In [ ]:
def crear_dataset(df_split, shuffle=False):
    img_paths = df_split["img_final"].values
    mask_paths = df_split["mask_final"].values

    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df_split), seed=RANDOM_STATE)

    ds = ds.map(cargar_imagen_y_mascara, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
train_ds = crear_dataset(df_train, shuffle=True)
val_ds = crear_dataset(df_val, shuffle=False)
test_ds = crear_dataset(df_test, shuffle=False)

print("Datasets creados.")

In [ ]:
for imgs, masks in train_ds.take(1):
    print("Batch imágenes:", imgs.shape)
    print("Batch máscaras:", masks.shape)

    img = imgs[0].numpy()
    mask = masks[0].numpy().squeeze()

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Imagen")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Máscara")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    inter = tf.reduce_sum(y_true_f * y_pred_f)
    total = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)

    dice = (2.0 * inter + smooth) / (total + smooth)

    return dice

In [ ]:
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

In [ ]:
def bce_dice_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)

    d_loss = dice_loss(y_true, y_pred)

    return bce + d_loss

In [ ]:
def iou_coef(y_true, y_pred, smooth=1e-6):
    y_pred_bin = tf.cast(y_pred > 0.5, tf.float32)

    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred_bin, [-1])

    inter = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - inter

    iou = (inter + smooth) / (union + smooth)

    return iou

In [ ]:
def bloque_conv(x, filtros):
    x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
    return x

In [ ]:
def crear_unet(input_shape=(256, 256, 3)):
    entrada = layers.Input(input_shape)

    # Encoder
    c1 = bloque_conv(entrada, 16)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = bloque_conv(p1, 32)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = bloque_conv(p2, 64)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = bloque_conv(p3, 128)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Centro
    c5 = bloque_conv(p4, 256)

    # Decoder
    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.Concatenate()([u6, c4])
    c6 = bloque_conv(u6, 128)

    u7 = layers.UpSampling2D((2, 2))(c6)
    u7 = layers.Concatenate()([u7, c3])
    c7 = bloque_conv(u7, 64)

    u8 = layers.UpSampling2D((2, 2))(c7)
    u8 = layers.Concatenate()([u8, c2])
    c8 = bloque_conv(u8, 32)

    u9 = layers.UpSampling2D((2, 2))(c8)
    u9 = layers.Concatenate()([u9, c1])
    c9 = bloque_conv(u9, 16)

    salida = layers.Conv2D(1, 1, activation="sigmoid")(c9)

    model = keras.Model(entrada, salida)

    return model

In [ ]:
keras.backend.clear_session()

model = crear_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[
        dice_coef,
        iou_coef,
        "binary_accuracy"
    ]
)

model.summary()

In [ ]:
model_path = MODELS_DIR / "oral_seg_unet_light_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_dice_coef",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_dice_coef",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("Modelo ligero se guardará en:")
print(model_path)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
final_model_path = MODELS_DIR / "oral_seg_unet_light_final.keras"
model.save(str(final_model_path))

df_history = pd.DataFrame(history.history)

history_csv = TABLES_DIR / "oral_seg_unet_light_history.csv"
df_history.to_csv(history_csv, index=False)

print("Modelo final guardado en:")
print(final_model_path)

print("Histórico guardado en:")
print(history_csv)

display(df_history.tail())

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history["dice_coef"], label="train")
plt.plot(history.history["val_dice_coef"], label="val")
plt.title("Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history["iou_coef"], label="train")
plt.plot(history.history["val_iou_coef"], label="val")
plt.title("IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()

plt.tight_layout()

fig_path = FIGURES_DIR / "oral_seg_unet_light_training_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True)

print("Resultados test:")
for k, v in test_results.items():
    print(k, ":", v)

In [ ]:
df_resumen = pd.DataFrame([{
    "modelo": "U-Net light",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "test_loss": test_results["loss"],
    "test_dice": test_results["dice_coef"],
    "test_iou": test_results["iou_coef"],
    "test_binary_accuracy": test_results["binary_accuracy"]
}])

csv_resumen = TABLES_DIR / "oral_seg_unet_light_resultados_test.csv"
df_resumen.to_csv(csv_resumen, index=False)

display(df_resumen)
print("Resumen guardado en:", csv_resumen)

In [ ]:
muestras = df_test.sample(
    n=min(8, len(df_test)),
    random_state=RANDOM_STATE
).reset_index(drop=True)

plt.figure(figsize=(12, 12))

for i, fila in muestras.iterrows():
    img_path = fila["img_final"]
    mask_path = fila["mask_final"]

    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    mask_real = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask_real is None:
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_norm = img_res.astype(np.float32) / 255.0

    pred = model.predict(np.expand_dims(img_norm, axis=0), verbose=0)[0]
    pred_mask = (pred.squeeze() > 0.5).astype(np.uint8) * 255

    mask_real_res = cv2.resize(mask_real, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    plt.subplot(4, 4, 2*i + 1)
    plt.imshow(img_res)
    plt.imshow(mask_real_res, cmap="jet", alpha=0.35)
    plt.title("Real " + fila["clase_lengua"], fontsize=8)
    plt.axis("off")

    plt.subplot(4, 4, 2*i + 2)
    plt.imshow(img_res)
    plt.imshow(pred_mask, cmap="jet", alpha=0.35)
    plt.title("Predicción", fontsize=8)
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "oral_seg_unet_light_predicciones_test.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
def dice_numpy(mask_real, mask_pred):
    real = (mask_real > 0).astype(np.uint8)
    pred = (mask_pred > 0).astype(np.uint8)

    inter = np.sum(real * pred)
    total = np.sum(real) + np.sum(pred)

    if total == 0:
        return 1.0

    return (2 * inter) / total

In [ ]:
def iou_numpy(mask_real, mask_pred):
    real = (mask_real > 0).astype(np.uint8)
    pred = (mask_pred > 0).astype(np.uint8)

    inter = np.sum(real * pred)
    union = np.sum(real) + np.sum(pred) - inter

    if union == 0:
        return 1.0

    return inter / union

In [ ]:
filas_metricas = []

for i, fila in df_test.iterrows():
    img_path = fila["img_final"]
    mask_path = fila["mask_final"]

    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    mask_real = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask_real is None:
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_norm = img_res.astype(np.float32) / 255.0

    pred = model.predict(np.expand_dims(img_norm, axis=0), verbose=0)[0]
    pred_mask = (pred.squeeze() > 0.5).astype(np.uint8) * 255

    mask_real_res = cv2.resize(mask_real, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    d = dice_numpy(mask_real_res, pred_mask)
    j = iou_numpy(mask_real_res, pred_mask)

    filas_metricas.append({
        "id_global": fila["id_global"],
        "origen": fila["origen"],
        "clase_lengua": fila["clase_lengua"],
        "dice": d,
        "iou": j,
        "filename_original": fila["filename_original"]
    })

df_metricas = pd.DataFrame(filas_metricas)

print("Métricas calculadas:", len(df_metricas))
display(df_metricas.head())

In [ ]:
print("Media global:")
display(df_metricas[["dice", "iou"]].mean())

print("Por origen:")
display(df_metricas.groupby("origen")[["dice", "iou"]].mean())

print("Por clase:")
display(df_metricas.groupby("clase_lengua")[["dice", "iou"]].mean())

print("Por origen y clase:")
display(df_metricas.groupby(["origen", "clase_lengua"])[["dice", "iou"]].mean())

In [ ]:
csv_metricas = TABLES_DIR / "oral_seg_unet_light_metricas_por_imagen.csv"
df_metricas.to_csv(csv_metricas, index=False)

print("Métricas guardadas en:", csv_metricas)